# Week 23: Next-Gen AI Reasoning, MCTS & Test-Time Compute
## Notebook 02: Process Reward Models (PRM) vs. Outcome Reward Models (ORM)

### 1. Conceptual Intuition & Real-World Use Case
In complex multi-step reasoning (Olympiad Math, Logic, Software Architecture):
- **Outcome Supervision (ORM):** Only checks if the final answer is correct ($+1$) or wrong ($-1$).
  - *Catastrophic Flaw:* If a student makes two algebraic errors that cancel out and accidentally lands on the right final number, the ORM rewards the broken logic!
- **Process Supervision (PRM):** Scores **every single intermediate reasoning step**!
  - Rewards sound logic even if the final calculation has an arithmetic typo.
  - Pinpoints the exact line where reasoning derailed.

**Lightman et al. (OpenAI, 2023) showed that PRMs dramatically outperform ORMs on the MATH benchmark!**

---

### 2. Mathematical Foundations: Step-Level Scoring
Let a reasoning trajectory be $S = (s_1, s_2, \dots, s_K)$ culminating in final answer $y$.

1. **Outcome Reward Model (ORM):**
   $$R_{\text{ORM}}(S) = P(\text{Correct}(y) \mid S)$$
2. **Process Reward Model (PRM):**
   Assigns a validity probability to each step $k$:
   $$r_k = P(\text{Step } k \text{ is mathematically sound} \mid s_1, \dots, s_k)$$
   Path Reliability Score:
   $$R_{\text{PRM}}(S) = \prod_{k=1}^K r_k \quad \text{or} \quad \min_{k} r_k$$

In [ ]:
import numpy as np

# Step 1: Compare Two Reasoning Paths for: "Solve 2x + 6 = 14"
# Path A: Correct reasoning throughout
path_A = [
    ("Step 1: Subtract 6 from both sides -> 2x = 8", 0.98),
    ("Step 2: Divide both sides by 2 -> x = 4", 0.99)
]

# Path B: Flawed reasoning with false positive final answer
path_B = [
    ("Step 1: Divide by 2 incorrectly -> x + 6 = 7", 0.12), # Critical reasoning bug!
    ("Step 2: Subtract 3 -> x = 4", 0.15)                   # Flawed step leading to accidental "correct" 4
]

# Step 2: Compare ORM vs PRM Scoring
def evaluate_orm(final_answer, true_answer=4):
    # ORM only checks final number!
    return 1.0 if final_answer == true_answer else 0.0

def evaluate_prm(path_steps):
    # PRM computes minimum step confidence across reasoning chain
    step_probs = [step[1] for step in path_steps]
    return min(step_probs), np.prod(step_probs)

orm_A = evaluate_orm(4)
orm_B = evaluate_orm(4)

prm_min_A, prm_prod_A = evaluate_prm(path_A)
prm_min_B, prm_prod_B = evaluate_prm(path_B)

print("--- Outcome Reward Model (ORM) vs Process Reward Model (PRM) ---")
print(f"Path A (Sound Logic):")
print(f"  ORM Score:  {orm_A:.1f} (Approved)")
print(f"  PRM Score:  Min Step = {prm_min_A:.2f}, Product = {prm_prod_A:.2f} (Sound ✅)")

print(f"\nPath B (Flawed Reasoning with Accidental Correct Answer):")
print(f"  ORM Score:  {orm_B:.1f} (FAILS TO CATCH ERROR! Approves broken logic!)")
print(f"  PRM Score:  Min Step = {prm_min_B:.2f}, Product = {prm_prod_B:.2f} (CAUGHT HALLUCINATION ❌)")

### 3. Step-by-Step Code Walkthrough
1. **The ORM Blindspot:** Path B arrived at $x=4$ through faulty logic, yet received a perfect 1.0 from the ORM.
2. **PRM Verification:** The PRM evaluated step 1 ($P=0.12$), immediately identifying the step as invalid and rejecting the chain.
3. **Credit Assignment:** PRMs solve the credit assignment problem by pinpointing the exact step where hallucinations began.

---

### 4. Industry Applications & Trade-offs
- **OpenAI o1 & PRM800K:** Automated training of reasoning models using millions of step-level verification labels.
- **Trade-off:** Data annotation cost: collecting human or synthetic step-by-step labels is substantially more expensive than checking final outcomes.

---

### 5. Top Interview Questions & Answers
**Q1: Why does Process Supervision generalize better to out-of-distribution reasoning than Outcome Supervision?**
> **Answer:** Outcome supervision rewards shortcuts and spurious correlations that happen to yield the right answer on training data. Process supervision enforces deductive step-by-step validity, preventing error cascading and reinforcing sound logical deduction.

**Q2: What is the "Best-of-N" sampling strategy using a PRM?**
> **Answer:** The generator LLM samples $N$ independent chain-of-thought solutions (e.g. $N=64$). The PRM scores every step of each candidate. The path with the highest minimum step score is selected, significantly boosting accuracy without fine-tuning the base generator.